# 🛰️ ORBIT-SHIELD — Pipeline de Machine Learning
## Detecção de Cyberataques em Ground Stations Satelitais

**Global Solution 2026.1 — FIAP**  
**Disciplinas:** Machine Learning & Data Science + Cibersegurança para IA

---

### 🎯 Objetivo
Construir um pipeline completo de Data Science capaz de:
1. **Detectar anomalias** no tráfego de rede de uma ground station (Isolation Forest)
2. **Classificar o tipo de ataque** quando uma anomalia é confirmada (Random Forest)
3. **Proteger o próprio modelo** contra Data Poisoning (validação estatística de entrada)

### 📡 Contexto Real
Em fevereiro de 2022, um cyberataque à rede de satélites ViaSat KA-SAT destruiu
milhares de modems em toda a Europa, comprometendo comunicações militares e civis.
O vetor de ataque foi a ground station — exatamente o que este sistema protege.

### 🗂️ Dataset
Usamos dados **simulados** com base nas características do **CICIDS2017**
(Canadian Institute for Cybersecurity Intrusion Detection System 2017),
um dataset de referência acadêmica mundial para detecção de intrusão.
Os dados são gerados programaticamente para garantir reprodutibilidade total.

---
## 📦 CÉLULA 1 — Instalação e Importação de Bibliotecas

**O que fazemos:** Importamos todas as bibliotecas necessárias.  
**Por que essas bibliotecas:**
- `pandas` / `numpy` → manipulação e operações numéricas sobre os dados
- `matplotlib` / `seaborn` → visualizações da EDA e métricas
- `scikit-learn` → modelos de ML (Isolation Forest, Random Forest) e métricas
- `scipy` → KS-Test para detecção de Data Poisoning

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 1: Importações
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

from scipy import stats
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    ConfusionMatrixDisplay
)

# Configurações globais de visualização
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
np.random.seed(42)  # Reprodutibilidade garantida

print('✅ Bibliotecas carregadas com sucesso.')
print(f'   NumPy: {np.__version__} | Pandas: {pd.__version__}')

---
## 🏭 CÉLULA 2 — Geração do Dataset Simulado

**O que fazemos:** Geramos um dataset sintético que simula o tráfego de rede
de uma ground station satelital, com tráfego normal e 4 tipos de ataque.

**Por que dados simulados:**  
Datasets reais de ground stations são classificados. A simulação baseada
nas distribuições estatísticas do CICIDS2017 é uma prática acadêmica aceita
e permite controle total sobre as características do experimento.

**Features geradas (inspiradas no CICIDS2017):**
- `duracao_conexao` → duração da sessão de rede em segundos
- `bytes_enviados` → volume de dados enviados (bytes)
- `bytes_recebidos` → volume de dados recebidos (bytes)
- `pacotes_por_segundo` → taxa de pacotes na conexão
- `flags_tcp` → número de flags TCP anômalos
- `tentativas_auth` → tentativas de autenticação na sessão
- `portas_destino_unicas` → diversidade de portas acessadas
- `intervalo_medio_pacotes` → tempo médio entre pacotes (ms)
- `tamanho_medio_pacote` → tamanho médio dos pacotes (bytes)
- `label` → NORMAL, DDOS, PORTSCAN, BRUTEFORCE, POISONING

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 2: Geração do Dataset
# ============================================================

def gerar_dataset_ground_station(n_normal=3000, seed=42):
    """
    Gera dataset sintético simulando tráfego de rede de uma
    ground station satelital com tráfego normal e 4 tipos de ataque.
    
    Parâmetros:
        n_normal (int): número de amostras de tráfego normal
        seed (int): semente para reprodutibilidade
    
    Retorna:
        pd.DataFrame: dataset completo com labels
    """
    rng = np.random.default_rng(seed)
    registros = []

    # ----------------------------------------------------------
    # CLASSE 0: Tráfego NORMAL da ground station
    # Características: conexões estáveis, baixa variância
    # ----------------------------------------------------------
    for _ in range(n_normal):
        registros.append({
            'duracao_conexao':         rng.normal(120, 30),
            'bytes_enviados':          rng.normal(50000, 10000),
            'bytes_recebidos':         rng.normal(80000, 15000),
            'pacotes_por_segundo':     rng.normal(50, 10),
            'flags_tcp':               rng.integers(0, 3),
            'tentativas_auth':         rng.integers(1, 3),
            'portas_destino_unicas':   rng.integers(1, 5),
            'intervalo_medio_pacotes': rng.normal(20, 5),
            'tamanho_medio_pacote':    rng.normal(512, 64),
            'label': 'NORMAL'
        })

    # ----------------------------------------------------------
    # CLASSE 1: Ataque DDoS (Distributed Denial of Service)
    # Características: altíssima taxa de pacotes, conexões curtas
    # Contramedida no sistema: Rate Limiting na API (Pilar Cyber)
    # ----------------------------------------------------------
    for _ in range(int(n_normal * 0.15)):
        registros.append({
            'duracao_conexao':         rng.normal(2, 1),
            'bytes_enviados':          rng.normal(200000, 50000),
            'bytes_recebidos':         rng.normal(5000, 1000),
            'pacotes_por_segundo':     rng.normal(5000, 500),
            'flags_tcp':               rng.integers(8, 15),
            'tentativas_auth':         rng.integers(1, 3),
            'portas_destino_unicas':   rng.integers(1, 3),
            'intervalo_medio_pacotes': rng.normal(0.2, 0.1),
            'tamanho_medio_pacote':    rng.normal(64, 10),
            'label': 'DDOS'
        })

    # ----------------------------------------------------------
    # CLASSE 2: Port Scan (reconhecimento da infraestrutura)
    # Características: muitas portas, pacotes pequenos e rápidos
    # ----------------------------------------------------------
    for _ in range(int(n_normal * 0.12)):
        registros.append({
            'duracao_conexao':         rng.normal(0.5, 0.2),
            'bytes_enviados':          rng.normal(200, 50),
            'bytes_recebidos':         rng.normal(100, 30),
            'pacotes_por_segundo':     rng.normal(200, 50),
            'flags_tcp':               rng.integers(1, 4),
            'tentativas_auth':         rng.integers(1, 2),
            'portas_destino_unicas':   rng.integers(100, 1000),
            'intervalo_medio_pacotes': rng.normal(5, 2),
            'tamanho_medio_pacote':    rng.normal(40, 5),
            'label': 'PORTSCAN'
        })

    # ----------------------------------------------------------
    # CLASSE 3: Brute Force (tentativas de login na estação)
    # Características: muitas tentativas de auth, tráfego repetitivo
    # ----------------------------------------------------------
    for _ in range(int(n_normal * 0.10)):
        registros.append({
            'duracao_conexao':         rng.normal(300, 60),
            'bytes_enviados':          rng.normal(30000, 5000),
            'bytes_recebidos':         rng.normal(20000, 3000),
            'pacotes_por_segundo':     rng.normal(80, 20),
            'flags_tcp':               rng.integers(2, 6),
            'tentativas_auth':         rng.integers(50, 200),
            'portas_destino_unicas':   rng.integers(1, 3),
            'intervalo_medio_pacotes': rng.normal(12, 3),
            'tamanho_medio_pacote':    rng.normal(256, 32),
            'label': 'BRUTEFORCE'
        })

    # ----------------------------------------------------------
    # CLASSE 4: Data Poisoning (praga virtual mais sofisticada)
    # CONEXÃO COM CIBERSEGURANÇA: Ataque STRIDE tipo Tampering
    # Características: SIMILAR ao normal, mas com desvios sutis
    # O perigo: regras simples não detectam — só ML detecta
    # ----------------------------------------------------------
    for _ in range(int(n_normal * 0.08)):
        registros.append({
            'duracao_conexao':         rng.normal(115, 35),    # Ligeiramente diferente do normal
            'bytes_enviados':          rng.normal(52000, 12000),
            'bytes_recebidos':         rng.normal(78000, 18000),
            'pacotes_por_segundo':     rng.normal(48, 15),
            'flags_tcp':               rng.integers(0, 5),     # Flags levemente elevados
            'tentativas_auth':         rng.integers(1, 4),
            'portas_destino_unicas':   rng.integers(1, 8),
            'intervalo_medio_pacotes': rng.normal(22, 8),
            'tamanho_medio_pacote':    rng.normal(510, 80),
            'label': 'POISONING'
        })

    df = pd.DataFrame(registros)
    # Garantir que não há valores negativos em features físicas
    colunas_positivas = ['duracao_conexao', 'bytes_enviados', 'bytes_recebidos',
                         'pacotes_por_segundo', 'intervalo_medio_pacotes',
                         'tamanho_medio_pacote']
    for col in colunas_positivas:
        df[col] = df[col].clip(lower=0.01)

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


# Gerar o dataset
df = gerar_dataset_ground_station(n_normal=3000)

print('✅ Dataset gerado com sucesso!')
print(f'   Total de registros: {len(df):,}')
print(f'   Features: {df.shape[1] - 1}')
print()
print('Distribuição de classes:')
print(df['label'].value_counts().to_string())

---
## 🔍 CÉLULA 3 — Análise Exploratória de Dados (EDA)

**O que fazemos:** Investigamos a estrutura, qualidade e distribuição dos dados
antes de qualquer modelagem. A EDA é obrigatória em qualquer pipeline sério de DS.

**Por que EDA rigorosa:**  
Modelos de ML treinados em dados mal compreendidos produzem resultados
enganosos — um risco crítico em sistemas de segurança onde um falso negativo
significa um ataque não detectado.

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 3: EDA — Visão Geral
# ============================================================

print('=' * 60)
print('ANÁLISE EXPLORATÓRIA DE DADOS — ORBIT-SHIELD')
print('=' * 60)

# 3.1 Estrutura do dataset
print('\n📋 Primeiras linhas do dataset:')
display(df.head())

# 3.2 Informações de tipos e nulos
print('\n📊 Informações gerais:')
display(df.info())

# 3.3 Estatísticas descritivas
print('\n📈 Estatísticas descritivas:')
display(df.describe().round(2))

# 3.4 Verificação de valores nulos
nulos = df.isnull().sum()
print(f'\n🔎 Valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else '   ✅ Nenhum valor nulo encontrado.')

# 3.5 Verificação de duplicatas
duplicatas = df.duplicated().sum()
print(f'\n🔁 Registros duplicados: {duplicatas}')
if duplicatas > 0:
    df = df.drop_duplicates()
    print(f'   ✅ Duplicatas removidas. Novo total: {len(df):,}')

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 3b: EDA — Visualizações
# ============================================================

features_numericas = [
    'duracao_conexao', 'bytes_enviados', 'pacotes_por_segundo',
    'tentativas_auth', 'portas_destino_unicas', 'intervalo_medio_pacotes'
]

# --- Gráfico 1: Distribuição de classes ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contagem = df['label'].value_counts()
cores = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#e67e22']

axes[0].bar(contagem.index, contagem.values, color=cores, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribuição de Classes no Dataset', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tipo de Tráfego')
axes[0].set_ylabel('Quantidade de Amostras')
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(contagem.values, labels=contagem.index, autopct='%1.1f%%',
            colors=cores, startangle=90, pctdistance=0.85)
axes[1].set_title('Proporção de Classes (%)', fontsize=13, fontweight='bold')

plt.suptitle('🛰️ ORBIT-SHIELD — Composição do Dataset', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 1 salvo: eda_distribuicao_classes.png')

# --- Gráfico 2: Boxplots por classe ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feature in enumerate(features_numericas):
    df.boxplot(column=feature, by='label', ax=axes[i],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('🔍 Distribuição das Features por Tipo de Ataque', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 2 salvo: eda_boxplots_features.png')

# --- Gráfico 3: Mapa de correlação ---
plt.figure(figsize=(10, 8))
corr = df[features_numericas + ['portas_destino_unicas']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('🔗 Mapa de Correlação entre Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico 3 salvo: eda_correlacao.png')

---
## ⚙️ CÉLULA 4 — Pré-processamento e Engenharia de Features

**O que fazemos:** Preparamos os dados para os modelos de ML.

**Decisões técnicas:**
- `StandardScaler` → normaliza as features para média 0 e desvio padrão 1.
  Necessário porque features com escalas muito diferentes (ex: bytes vs flags)
  podem dominar incorretamente o aprendizado do modelo
- **Feature Engineering:** criamos 2 features derivadas que capturam
  comportamentos suspeitos não óbvios nas features originais
- `LabelEncoder` → converte labels texto em números para o Random Forest

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 4: Pré-processamento
# ============================================================

# 4.1 Engenharia de Features
# Criamos features derivadas que amplificam padrões de ataque

df_proc = df.copy()

# Feature 1: Ratio bytes enviados/recebidos
# DDoS envia muito mais do que recebe → ratio muito alto
# Conexões normais têm ratio próximo de 1
df_proc['ratio_bytes'] = (
    df_proc['bytes_enviados'] / (df_proc['bytes_recebidos'] + 1)
).round(4)

# Feature 2: Score de suspeita de autenticação
# Brute Force: muitas tentativas em pouco tempo = alta taxa
df_proc['taxa_auth_por_segundo'] = (
    df_proc['tentativas_auth'] / (df_proc['duracao_conexao'] + 0.001)
).round(4)

print('✅ Features derivadas criadas:')
print('   → ratio_bytes: detecta DDoS e exfiltração de dados')
print('   → taxa_auth_por_segundo: detecta Brute Force')

# 4.2 Separação entre features (X) e target (y)
feature_cols = [
    'duracao_conexao', 'bytes_enviados', 'bytes_recebidos',
    'pacotes_por_segundo', 'flags_tcp', 'tentativas_auth',
    'portas_destino_unicas', 'intervalo_medio_pacotes',
    'tamanho_medio_pacote', 'ratio_bytes', 'taxa_auth_por_segundo'
]

X = df_proc[feature_cols].copy()
y = df_proc['label'].copy()

# 4.3 Codificação das labels para o modelo supervisionado
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f'\n📊 Mapeamento de classes:')
for i, classe in enumerate(le.classes_):
    print(f'   {i} → {classe}')

# 4.4 Divisão treino/teste (80% / 20%)
# Estratificada: garante proporção igual de classes em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4.5 Normalização
# IMPORTANTE: o scaler é treinado APENAS no treino para evitar data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)     # transform() — não fit_transform()

print(f'\n✅ Divisão treino/teste concluída:')
print(f'   Treino: {X_train_scaled.shape[0]:,} amostras')
print(f'   Teste:  {X_test_scaled.shape[0]:,} amostras')
print(f'   Features totais: {X_train_scaled.shape[1]}')

---
## 🤖 CÉLULA 5 — Modelo 1: Isolation Forest (Detecção de Anomalia)

**O que fazemos:** Treinamos o Isolation Forest para identificar tráfego anômalo.

**Por que Isolation Forest:**  
- Algoritmo **não supervisionado** — não precisa de exemplos rotulados de ataque
- Funciona isolando pontos de dados: anomalias são isoladas em menos partições
- Perfeito para segurança: ataques zero-day (nunca vistos antes) também são detectados
- `contamination=0.1` → estimamos que ~10% do tráfego é anômalo (soma dos ataques)

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 5: Isolation Forest
# ============================================================

print('🔄 Treinando Isolation Forest...')

# Isolation Forest detecta anomalias sem precisar das labels
# contamination: proporção esperada de anomalias no dataset
iso_forest = IsolationForest(
    n_estimators=200,      # Número de árvores — mais árvores = mais estável
    contamination=0.10,    # ~10% dos dados são ataques (DDoS + PortScan + BruteForce + Poisoning)
    random_state=42,
    n_jobs=-1              # Usa todos os núcleos disponíveis
)

iso_forest.fit(X_train_scaled)

# Predição: 1 = normal, -1 = anomalia
y_pred_iso = iso_forest.predict(X_test_scaled)

# Converter para binário: 0 = normal, 1 = anomalia
y_pred_iso_bin = (y_pred_iso == -1).astype(int)

# Criar ground truth binária: 0 = NORMAL, 1 = qualquer ataque
y_test_bin = (y_test != le.transform(['NORMAL'])[0]).astype(int)

# --- Métricas do Isolation Forest ---
acc_iso  = accuracy_score(y_test_bin, y_pred_iso_bin)
prec_iso = precision_score(y_test_bin, y_pred_iso_bin, zero_division=0)
rec_iso  = recall_score(y_test_bin, y_pred_iso_bin, zero_division=0)
f1_iso   = f1_score(y_test_bin, y_pred_iso_bin, zero_division=0)

print('\n' + '=' * 50)
print('📊 MÉTRICAS — ISOLATION FOREST')
print('=' * 50)
print(f'   Acurácia:  {acc_iso:.4f} ({acc_iso*100:.2f}%)')
print(f'   Precision: {prec_iso:.4f}  ← % ataques reais entre os alertas emitidos')
print(f'   Recall:    {rec_iso:.4f}  ← % ataques reais que foram detectados')
print(f'   F1-Score:  {f1_iso:.4f}  ← média harmônica Precision/Recall')
print()
print('⚠️  Em segurança, Recall é a métrica mais crítica:')
print('    Recall baixo = ataques reais não detectados = sistema falhou.')

# --- Visualização: Confusion Matrix ---
fig, ax = plt.subplots(figsize=(7, 5))
cm_iso = confusion_matrix(y_test_bin, y_pred_iso_bin)
disp = ConfusionMatrixDisplay(cm_iso, display_labels=['Normal', 'Ataque'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('🌲 Isolation Forest — Matriz de Confusão\n(Detecção: Normal vs Ataque)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('modelo1_isolation_forest_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: modelo1_isolation_forest_cm.png')

# --- Score de anomalia (quanto mais negativo = mais suspeito) ---
scores_anomalia = iso_forest.score_samples(X_test_scaled)

plt.figure(figsize=(12, 4))
plt.hist(scores_anomalia[y_test_bin == 0], bins=50, alpha=0.6,
         color='#2ecc71', label='Tráfego Normal')
plt.hist(scores_anomalia[y_test_bin == 1], bins=50, alpha=0.6,
         color='#e74c3c', label='Tráfego Malicioso')
plt.axvline(x=np.percentile(scores_anomalia, 10), color='black',
            linestyle='--', linewidth=2, label='Threshold (10%)')
plt.xlabel('Score de Anomalia (mais negativo = mais suspeito)')
plt.ylabel('Frequência')
plt.title('🎯 Separação de Scores: Normal vs Ataque', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('modelo1_scores_anomalia.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: modelo1_scores_anomalia.png')

---
## 🌲 CÉLULA 6 — Modelo 2: Random Forest (Classificação do Tipo de Ataque)

**O que fazemos:** Treinamos o Random Forest para classificar o tipo específico
de ataque (DDoS, PortScan, BruteForce, Poisoning ou Normal).

**Por que Random Forest:**
- Ensemble de árvores de decisão — robusto a overfitting
- Funciona bem com datasets de tamanho médio (adequado para 1º semestre)
- Fornece `feature_importance` — explica quais features mais importaram
- Interpretável: podemos mostrar à banca qual feature "delatou" cada ataque

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 6: Random Forest
# ============================================================

print('🔄 Treinando Random Forest...')

rf = RandomForestClassifier(
    n_estimators=200,       # 200 árvores — bom balanço entre performance e tempo
    max_depth=15,           # Limita profundidade para evitar overfitting
    min_samples_split=5,    # Nó precisa de ao menos 5 amostras para dividir
    class_weight='balanced',# Compensa o desbalanceamento de classes
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

# --- Métricas detalhadas por classe ---
print('\n' + '=' * 60)
print('📊 MÉTRICAS — RANDOM FOREST (Classificação Multiclasse)')
print('=' * 60)
print(classification_report(
    y_test, y_pred_rf,
    target_names=le.classes_,
    digits=4
))

# Métricas gerais
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf, average='weighted')
print(f'   Acurácia Geral:   {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'   F1-Score (macro): {f1_rf:.4f}')

# Validação cruzada (5-fold)
cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='f1_weighted')
print(f'\n   Cross-Validation F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'   Interpretação: modelo estável (baixo desvio = não overfitting)')

# --- Matriz de Confusão Multiclasse ---
fig, ax = plt.subplots(figsize=(9, 7))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predito', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title('🌲 Random Forest — Matriz de Confusão\n(Classificação de Tipos de Ataque)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('modelo2_random_forest_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: modelo2_random_forest_cm.png')

# --- Importância das Features ---
importancias = pd.Series(rf.feature_importances_, index=feature_cols)
importancias = importancias.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
cores_imp = ['#e74c3c' if i >= len(importancias)-3 else '#3498db'
             for i in range(len(importancias))]
importancias.plot(kind='barh', color=cores_imp, edgecolor='white')
plt.xlabel('Importância Relativa', fontsize=11)
plt.title('🏆 Importância das Features — Random Forest\n(Vermelho = Top 3 mais importantes)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('modelo2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: modelo2_feature_importance.png')
print('\n🔍 Top 3 features mais importantes:')
print(importancias.tail(3).to_string())

---
## 🛡️ CÉLULA 7 — Defesa contra Data Poisoning (Pilar Cibersegurança)

**O que fazemos:** Implementamos a contramedida técnica contra o ataque de
Data Poisoning — o mais sofisticado do mapa STRIDE do ORBIT-SHIELD.

**Conexão direta com Cibersegurança:**  
Esta célula implementa a contramedida para a ameaça #6 do nosso modelo STRIDE
(Elevation of Privilege via envenenamento do modelo de ML).

**Técnica usada — Kolmogorov-Smirnov Test (KS-Test):**  
Compara estatisticamente a distribuição dos dados de entrada com a distribuição
dos dados de treino certificados. Se os dados novos forem estatisticamente
diferentes, o sistema rejeita o lote e emite um alerta de poisoning.

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 7: Defesa contra Data Poisoning
# ============================================================

class DetectorDataPoisoning:
    """
    Sistema de defesa contra ataques de Data Poisoning.
    
    Utiliza o teste de Kolmogorov-Smirnov para comparar
    estatisticamente a distribuição de novos dados com
    a distribuição de referência dos dados certificados.
    
    CONEXÃO COM CIBERSEGURANÇA:
    Implementa a contramedida para a ameaça STRIDE #6:
    Elevation of Privilege via envenenamento do modelo de ML.
    """

    def __init__(self, alpha: float = 0.05, limiar_features: float = 0.3):
        """
        Parâmetros:
            alpha: nível de significância estatística (padrão: 5%)
            limiar_features: % máximo de features comprometidas aceito
        """
        self.alpha = alpha
        self.limiar_features = limiar_features
        self.dados_referencia = None
        self.feature_names = None

    def calibrar(self, X_referencia: np.ndarray, feature_names: list):
        """Registra os dados certificados como referência de distribuição normal."""
        self.dados_referencia = X_referencia
        self.feature_names = feature_names
        print(f'✅ Detector calibrado com {len(X_referencia):,} amostras de referência.')

    def analisar_lote(self, X_novo: np.ndarray, nome_lote: str = 'lote') -> dict:
        """
        Analisa um novo lote de dados quanto ao risco de Data Poisoning.
        
        Retorna dicionário com resultado da análise e features suspeitas.
        """
        if self.dados_referencia is None:
            raise RuntimeError('Detector não calibrado. Chame calibrar() primeiro.')

        features_suspeitas = []
        p_values = []

        for i, feature in enumerate(self.feature_names):
            # KS-Test: compara as distribuições cumulativas
            # H0: as amostras vêm da mesma distribuição
            # H1: as amostras vêm de distribuições diferentes
            stat, p_value = stats.ks_2samp(
                self.dados_referencia[:, i],
                X_novo[:, i]
            )
            p_values.append(p_value)
            if p_value < self.alpha:  # Rejeita H0: distribuição alterada
                features_suspeitas.append((feature, p_value, stat))

        proporcao_suspeita = len(features_suspeitas) / len(self.feature_names)
        envenenado = proporcao_suspeita >= self.limiar_features

        resultado = {
            'lote': nome_lote,
            'envenenado': envenenado,
            'nivel_alerta': '🔴 CRÍTICO' if envenenado else '🟢 SEGURO',
            'features_suspeitas': features_suspeitas,
            'proporcao_suspeita': proporcao_suspeita,
            'p_values': dict(zip(self.feature_names, p_values))
        }
        return resultado


# --- Instanciar e calibrar o detector ---
detector = DetectorDataPoisoning(alpha=0.05, limiar_features=0.30)
detector.calibrar(X_train_scaled, feature_cols)

print()

# --- Teste 1: Lote de dados normais (deve passar) ---
X_dados_normais = X_test_scaled[y_test == le.transform(['NORMAL'])[0]][:100]
resultado_normal = detector.analisar_lote(X_dados_normais, 'lote_normal')

print('=' * 55)
print(f"📦 Análise: {resultado_normal['lote']}")
print(f"   Status: {resultado_normal['nivel_alerta']}")
print(f"   Features suspeitas: {len(resultado_normal['features_suspeitas'])}/{len(feature_cols)}")
print(f"   Proporção suspeita: {resultado_normal['proporcao_suspeita']:.1%}")

print()

# --- Teste 2: Lote envenenado (deve ser bloqueado) ---
rng = np.random.default_rng(99)
X_envenenado = X_test_scaled.copy()[:100]
# Simula injeção de valores alterados em 50% das features
for col_idx in range(0, X_envenenado.shape[1], 2):
    X_envenenado[:, col_idx] += rng.normal(3.5, 1.0, size=100)  # Deslocamento sistemático

resultado_poison = detector.analisar_lote(X_envenenado, 'lote_envenenado')

print('=' * 55)
print(f"📦 Análise: {resultado_poison['lote']}")
print(f"   Status: {resultado_poison['nivel_alerta']}")
print(f"   Features suspeitas: {len(resultado_poison['features_suspeitas'])}/{len(feature_cols)}")
print(f"   Proporção suspeita: {resultado_poison['proporcao_suspeita']:.1%}")
if resultado_poison['features_suspeitas']:
    print('   Features comprometidas:')
    for feat, pval, stat in resultado_poison['features_suspeitas'][:5]:
        print(f'     → {feat}: p-value={pval:.6f} (stat={stat:.4f})')

print()
print('🛡️  Sistema de defesa funcionando: lote envenenado bloqueado antes de contaminar o modelo.')

---
## 📊 CÉLULA 8 — Sumário Final e Exportação de Resultados

**O que fazemos:** Consolidamos todas as métricas, geramos o relatório final
e exportamos os resultados em CSV para o banco de dados e o dashboard.

In [ ]:
# ============================================================
# ORBIT-SHIELD | Célula 8: Sumário Final
# ============================================================

print('\n' + '🛰️ ' * 20)
print('       ORBIT-SHIELD — RELATÓRIO FINAL DE ML')
print('🛰️ ' * 20)

print(f"""
╔══════════════════════════════════════════════════════╗
║           MODELO 1: ISOLATION FOREST                 ║
║         (Detecção de Anomalia — Não Supervisionado)  ║
╠══════════════════════════════════════════════════════╣
║  Acurácia:   {acc_iso*100:.2f}%                               ║
║  Precision:  {prec_iso*100:.2f}%  (% alertas que são reais)  ║
║  Recall:     {rec_iso*100:.2f}%  (% ataques detectados)      ║
║  F1-Score:   {f1_iso*100:.2f}%                               ║
╚══════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════╗
║           MODELO 2: RANDOM FOREST                    ║
║       (Classificação de Tipo de Ataque — Sup.)       ║
╠══════════════════════════════════════════════════════╣
║  Acurácia:   {acc_rf*100:.2f}%                               ║
║  F1-Score:   {f1_rf*100:.2f}%  (weighted)                   ║
║  CV Score:   {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%  (5-fold)      ║
╚══════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════╗
║        DEFESA CONTRA DATA POISONING                  ║
╠══════════════════════════════════════════════════════╣
║  Lote Normal:     ✅ APROVADO                        ║
║  Lote Envenenado: 🔴 BLOQUEADO (KS-Test)             ║
╚══════════════════════════════════════════════════════╝
""")

# Exportar predições para CSV (integração com banco de dados)
df_resultados = pd.DataFrame({
    'indice': range(len(y_test)),
    'label_real': le.inverse_transform(y_test),
    'label_predita_rf': le.inverse_transform(y_pred_rf),
    'anomalia_iso': y_pred_iso_bin,
    'score_anomalia': iso_forest.score_samples(X_test_scaled).round(4)
})

df_resultados.to_csv('orbit_shield_predicoes.csv', index=False)
print('✅ Predições exportadas: orbit_shield_predicoes.csv')
print('   → Este arquivo alimenta o banco de dados e o dashboard.')
print()
print('🎯 Pipeline de ML concluído com sucesso!')
print('   Próxima etapa: Banco de Dados (DDL + queries de agregação)')